In [1]:
import sys

!{sys.executable} -m pip install ragas datasets


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ========================
# Imports
# ========================

import os

from langchain_huggingface import (
    HuggingFaceEmbeddings
)

from langchain_community.vectorstores import (
    Chroma
)

from langchain_mistralai import (
    ChatMistralAI
)

c:\Users\meytb\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ========================
# Mistral API Key
# ========================

os.environ["MISTRAL_API_KEY"] = (
    
    "mubazt0eFkAT8SBZM6vgkeLkYkJFPfs9"
)

In [4]:
# ========================
# Embedding Model
# ========================

embedding_model = (
    
    HuggingFaceEmbeddings(
        
        model_name=
        "sentence-transformers/all-MiniLM-L6-v2"
    )
)

print(
    "Embedding model loaded."
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2312.21it/s]


Embedding model loaded.


In [5]:
# ========================
# Load ChromaDB
# ========================

vectorstore = Chroma(
    
    persist_directory=
    "chroma_db",
    
    embedding_function=
    embedding_model
)

print(
    "ChromaDB loaded."
)

C:\Users\meytb\AppData\Local\Temp\ipykernel_21320\570944923.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


ChromaDB loaded.


In [6]:
# ========================
# Retriever
# ========================

retriever = (
    
    vectorstore.as_retriever(
        
        search_kwargs={
            
            "k": 5
        }
    )
)

print(
    "Retriever created."
)

Retriever created.


In [7]:
# ========================
# Mistral LLM
# ========================

llm = ChatMistralAI(
    
    model=
    "mistral-large-latest",
    
    temperature=0.2
)

print(
    "Mistral connected."
)

Mistral connected.


In [8]:
from langchain_core.prompts import (
    ChatPromptTemplate
)

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_core.runnables import (
    RunnablePassthrough
)

In [9]:
# ========================
# Financial Prompt
# ========================

prompt_template = """
You are FinShield AI,
an expert financial risk assistant.

Use ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = (
    
    ChatPromptTemplate.from_template(
        
        prompt_template
    )
)

In [10]:
# ========================
# Format Retrieved Docs
# ========================

def format_docs(docs):
    
    return "\n\n".join(
        
        doc.page_content
        
        for doc in docs
    )

In [11]:
# ========================
# RAG Chain
# ========================

rag_chain = (
    
    {
        "context":
        retriever | format_docs,
        
        "question":
        RunnablePassthrough()
    }
    
    | prompt
    
    | llm
    
    | StrOutputParser()
)

print(
    "RAG chain created."
)

RAG chain created.


# Build RAG Evaluation Dataset

This section creates a benchmark dataset used to evaluate the financial RAG assistant.

The evaluation dataset contains:
- financial questions
- retrieved contexts
- generated answers
- expected ground truth answers

In [12]:
# ========================
# Evaluation Questions
# ========================

evaluation_data = [

    {
        "question":
        "Why do low EXT_SOURCE scores increase default risk?",
        
        "ground_truth":
        "Low EXT_SOURCE values are strongly associated with higher default probability and increased borrower risk."
    },
    
    {
        "question":
        "Why are false negatives dangerous in fraud detection?",
        
        "ground_truth":
        "False negatives allow fraudulent transactions to pass undetected, creating financial losses."
    }
]

print(
    f"Evaluation questions: "
    f"{len(evaluation_data)}"
)

Evaluation questions: 2


In [13]:
import time

questions = []

answers = []

contexts = []

ground_truths = []

for item in evaluation_data:
    
    question = item["question"]
    
    # Retrieve docs
    retrieved_docs = (
        
        retriever.invoke(question)
    )
    
    retrieved_contexts = [
        
        doc.page_content
        
        for doc in retrieved_docs
    ]
    
    # Generate answer
    response = (
        
        rag_chain.invoke(question)
    )
    
    # Store
    questions.append(question)
    
    answers.append(response)
    
    contexts.append(retrieved_contexts)
    
    ground_truths.append(
        
        item["ground_truth"]
    )
    
    # Avoid rate limit
    time.sleep(3)

print(
    "RAG answers generated."
)

RAG answers generated.


In [14]:
# ========================
# Preview Evaluation Sample
# ========================

print(
    "QUESTION:\n"
)

print(
    questions[0]
)

print(
    "\nANSWER:\n"
)

print(
    answers[0]
)

print(
    "\nGROUND TRUTH:\n"
)

print(
    ground_truths[0]
)

QUESTION:

Why do low EXT_SOURCE scores increase default risk?

ANSWER:

Based on the provided context, **low EXT_SOURCE scores (EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3) increase default risk** because these external scores are **strongly predictive variables** in the credit scoring system. Specifically:

1. **Predictive Influence**: The `EXT_SOURCE` variables are among the *most important* features for borrower risk predictions. Their values directly correlate with default probability—lower scores signal higher risk.
2. **Default Probability Link**: The context explicitly states that *"low EXT_SOURCE values generally increase default probability."* This suggests these scores likely reflect underlying risk factors (e.g., credit history, financial stability, or external bureau data) that historically align with borrower defaults.
3. **Model Interpretation**: Since the system relies on these scores to separate risky borrowers from safer ones (as measured by metrics like the Gini coeffic

# Compute RAGAS Metrics 

In [15]:
# ========================
# RAGAS Imports
# ========================

from datasets import Dataset

from ragas import evaluate

from ragas.metrics import (
    
    faithfulness,
    
    answer_relevancy,
    
    context_precision
)

C:\Users\meytb\AppData\Local\Temp\ipykernel_21320\2170329348.py:9: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\meytb\AppData\Local\Temp\ipykernel_21320\2170329348.py:9: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\meytb\AppData\Local\Temp\ipykernel_21320\2170329348.py:9: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (


In [16]:
# ========================
# Create RAGAS Dataset
# ========================

ragas_dataset = Dataset.from_dict(
    {
        "question": questions,
        
        "answer": answers,
        
        "contexts": contexts,
        
        "ground_truth": ground_truths
    }
)

print(
    ragas_dataset
)

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 2
})


In [17]:

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import (
    LangchainEmbeddingsWrapper
)
# ========================
# RAGAS Mistral Wrapper
# ========================

ragas_llm = (
    
    LangchainLLMWrapper(
        
        llm
    )
)

print(
    "RAGAS using Mistral."
)
# ========================
# RAGAS Embeddings Wrapper
# ========================

ragas_embeddings = (
    
    LangchainEmbeddingsWrapper(
        
        embedding_model
    )
)

print(
    "RAGAS embeddings configured."
)
# ========================
# Full RAGAS Evaluation
# ========================

results = evaluate(
    
    ragas_dataset,
    
    metrics=[
        
        faithfulness,
        
        answer_relevancy,
        
        context_precision
    ],
    
    llm=ragas_llm,
    
    embeddings=ragas_embeddings
)

print(results)

C:\Users\meytb\AppData\Local\Temp\ipykernel_21320\1773314556.py:11: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  LangchainLLMWrapper(
C:\Users\meytb\AppData\Local\Temp\ipykernel_21320\1773314556.py:26: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  LangchainEmbeddingsWrapper(


RAGAS using Mistral.
RAGAS embeddings configured.


Evaluating: 100%|██████████| 6/6 [03:00<00:00, 30.06s/it]


{'faithfulness': 0.5250, 'answer_relevancy': nan, 'context_precision': nan}
